# Unified Invoice OCR Pipeline
Jeden notebook obsługuje wszystkich dostawców. Konfiguracja per-dostawca w `config/<supplier>.json`, prompty w `prompts/<supplier>.txt`.

**Dostawcy:** `airspace` | `courier_network` | `easyflyer` | `xpd`

In [36]:
pip install -r requirements.txt --trusted-host pypi.org --trusted-host files.pythonhosted.org --trusted-host pypi.python.org


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from snowflake.snowpark import Session
import json
import os
import re
import uuid

import pandas as pd
from openpyxl import load_workbook

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
from PIL import Image, ImageEnhance
from pdf2image import convert_from_path

## Konfiguracja — wybierz dostawcę i ścieżki

In [51]:
# ── WYBIERZ DOSTAWCĘ ─────────────────────────────────────────────────────────
# Zmień tę wartość na: "airspace" | "courier_network" | "easyflyer" | "xpd"
SUPPLIER = "xpd"

# ── ŚCIEŻKI ───────────────────────────────────────────────────────────────────
BASE_DIR     = os.path.dirname(os.path.abspath("uni_pipeline.ipynb"))
CONFIG_DIR   = os.path.join(BASE_DIR, "config")
PROMPTS_DIR  = os.path.join(BASE_DIR, "prompts")
OCR_OUT_DIR  = os.path.join(BASE_DIR, "ocr_output", SUPPLIER)

# Tesseract executable path (Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler path for PDF conversion (Windows)
POPPLER_PATH = r"C:\Program Files\poppler\poppler-25.12.0\Library\bin"

# ── ZAŁADUJ KONFIGURACJĘ ──────────────────────────────────────────────────────
config_path = os.path.join(CONFIG_DIR, f"{SUPPLIER}.json")
with open(config_path, encoding="utf-8") as f:
    CFG = json.load(f)

INPUT_DIR    = CFG["input_directory"]
PROMPT_FILE  = os.path.join(BASE_DIR, CFG["prompt_file"])

print(f"Supplier      : {CFG['supplier_name']}")
print(f"Input dir     : {INPUT_DIR}")
print(f"Prompt file   : {PROMPT_FILE}")
print(f"chars_per_pixel: {CFG['ocr']['chars_per_pixel']}")
print(f"Tesseract PSM : {CFG['ocr']['tesseract_psm']}")

Supplier      : XPD Global
Input dir     : C:\Users\jakub.klosowski\OneDrive - Autoliv\Desktop\Invoices\Xpd
Prompt file   : c:\Users\jakub.klosowski\email\FAP\unified\prompts/xpd.txt
chars_per_pixel: 17
Tesseract PSM : 11


## Połączenie Snowflake

In [39]:
connection_parameters = {
    "user":          "JAKUB.KLOSOWSKI@AUTOLIV.COM",
    "authenticator": "externalbrowser",
    "account":       "VN02639-YR60386",
    "warehouse":     "DEV_PRINCIPAL_WH",
    "database":      "PROTO_DEVTEAM_DB",
    "schema":        "PUBLIC",
}

session = Session.builder.configs(connection_parameters).create()
print("Snowflake session created.")

Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://login.microsoftonline.com/f3603cca-3b48-4625-afd2-2eb0587bf1d6/saml2?SAMLRequest=nZJbc9owEIX%2Fikd9tiXb4IAGyFBopszk4nJJCm%2FCXhNNZMlIcpz011fGMJM%2BJA9988hn9zu7Z0fXb6XwXkEbruQYhQFBHshM5VwexmizvvEHyDOWyZwJJWGM3sGg68nIsFJUdFrbZ7mEYw3Geq6RNLT9MUa1llQxww2VrARDbUZX07tbGgWEMmNAW4dD55LccMd6traiGDdNEzRxoPQBR4QQTIbYqVrJN%2FQBUX3NqLSyKlPiUvLmZvoEEWLSaxFO4QjpufA7l90KvqLsO5GhP9fr1E8fVmvkTS%2FTzZQ0dQl6BfqVZ7BZ3nYGjHOwXSYkHiRB4%2FbmQ61VBQH7U2sIjFRNIdgLZKqsauu6B%2B4LF5BjoQ7c7WwxH6Pqhefb280x%2FX1cDncNf%2FxxfJLzJxbzO7Up5umv7c4we78v7ZY1s2mGvMdLwlGb8MKYGhayzdW6JxIlPon9KFmTKxoOaUyCfkx2yJs7f1wye6q8mD%2F5CEqeaWVUYZUUXELnMnZjZRnz431v4PeSqO%2BzIo%2F8CPakP7jaF2Ge4Da9CHUXRE9G9OR%2F9zLCH7ucj%2FLe5bSYp0rw7N27Ubpk9vMYwyA8vfDcL05SCiXjYprnGoxxcQqhmpkGZt3tW10DwpOO%2Bu%2F1T%2F4C&RelayState=ver%3A3-hint%3A3302906936078-ETMsDgAAAZ0pAzniABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEJ753WO%2BQyIKm7BIYwae8yUAAACQ4WKXzO

## Preprocessing obrazu — parametryczny

In [52]:
def deskew_image_pil(pil_img):
    """Deskew image using horizontal projection profile (-15° to +15°)."""
    img = np.array(pil_img.convert("L"))
    img_bin = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 15, 8)
    h, w = img_bin.shape
    center = (w // 2, h // 2)
    best_angle, best_score = 0.0, -1.0
    for angle in np.arange(-15, 15.1, 0.5):
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        rotated = cv2.warpAffine(img_bin, M, (w, h), flags=cv2.INTER_NEAREST, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        score = float(np.var(np.sum(rotated, axis=1)))
        if score > best_score:
            best_score, best_angle = score, angle
    M = cv2.getRotationMatrix2D(center, best_angle, 1.0)
    rotated_img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
    print(f"  Deskew angle: {best_angle:.1f}°")
    return Image.fromarray(rotated_img)


def preprocess_image_pil(img, pre_cfg):
    """
    Parametryczny preprocessing obrazu sterowany przez słownik pre_cfg z config/*.json.

    pre_cfg keys:
      contrast (float|null)       — ImageEnhance.Contrast factor; null = skip
      sharpness (float|null)      — ImageEnhance.Sharpness factor; null = skip
      scale (float|null)          — upscale factor (e.g. 1.5); null = skip
      threshold_method (str)      — "otsu" | "adaptive"
      threshold_block (int|null)  — adaptive only: block size (odd number)
      threshold_c (int|null)      — adaptive only: constant C
      deskew (bool)               — whether to run deskew before binarisation
    """
    if img.mode != "L":
        img = img.convert("L")

    if pre_cfg.get("deskew"):
        img = deskew_image_pil(img)

    if pre_cfg.get("contrast"):
        img = ImageEnhance.Contrast(img).enhance(pre_cfg["contrast"])

    if pre_cfg.get("sharpness"):
        img = ImageEnhance.Sharpness(img).enhance(pre_cfg["sharpness"])

    if pre_cfg.get("scale"):
        factor = pre_cfg["scale"]
        img = img.resize(
            (int(img.width * factor), int(img.height * factor)),
            Image.Resampling.LANCZOS,
        )

    img_np = np.array(img)

    if pre_cfg.get("threshold_method") == "adaptive":
        block = pre_cfg.get("threshold_block", 21)
        c_val = pre_cfg.get("threshold_c", 10)
        img_bin = cv2.adaptiveThreshold(
            img_np, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block, c_val
        )
    else:  # otsu (default)
        img_blur = cv2.GaussianBlur(img_np, (1, 1), 0)
        _, img_bin = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return Image.fromarray(img_bin)


print("preprocess_image_pil() ready")

preprocess_image_pil() ready


## OCR — pozycjonowany tekst

In [53]:
def _create_positioned_line(tokens, chars_per_pixel):
    tokens.sort(key=lambda x: x["x"])
    if not tokens:
        return ""
    first = tokens[0]
    result = " " * max(0, first["x"] // chars_per_pixel) + first["text"]
    current_x = first["x"] + len(first["text"]) * chars_per_pixel
    for item in tokens[1:]:
        spaces = max(1, (item["x"] - current_x) // chars_per_pixel)
        result += " " * spaces + item["text"]
        current_x = item["x"] + len(item["text"]) * chars_per_pixel
    return result


def extract_with_real_positions(image_input, ocr_cfg):
    """
    Run Tesseract OCR and reconstruct text preserving spatial layout.
    ocr_cfg keys: chars_per_pixel, tesseract_psm, tesseract_oem, dpi
    """
    cpp  = ocr_cfg.get("chars_per_pixel", 17)
    psm  = ocr_cfg.get("tesseract_psm", 4)
    oem  = ocr_cfg.get("tesseract_oem", 1)
    dpi  = ocr_cfg.get("dpi", 300)

    image = Image.open(image_input) if isinstance(image_input, str) else image_input
    config_str = f"--oem {oem} --psm {psm} --dpi {dpi}"

    data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT, config=config_str)

    tokens = [
        {"text": data["text"][i], "x": data["left"][i], "y": data["top"][i],
         "width": data["width"][i], "height": data["height"][i]}
        for i in range(len(data["text"]))
        if data["text"][i].strip()
    ]

    tokens.sort(key=lambda t: (t["y"], t["x"]))
    lines, current_line, current_y = [], [], None

    for token in tokens:
        if current_y is None or abs(token["y"] - current_y) < cpp:
            current_line.append(token)
            current_y = token["y"]
        else:
            lines.append(_create_positioned_line(current_line, cpp))
            current_line = [token]
            current_y = token["y"]

    if current_line:
        lines.append(_create_positioned_line(current_line, cpp))

    return "\n".join(lines)


def _pdf_to_images(pdf_path, ocr_cfg, max_pages=None):
    dpi = ocr_cfg.get("dpi", 300)
    kwargs = {"dpi": dpi, "poppler_path": POPPLER_PATH}
    if max_pages:
        kwargs.update({"first_page": 1, "last_page": max_pages})
    return convert_from_path(pdf_path, **kwargs)


def pdf_to_ocr_text(pdf_path, pre_cfg, ocr_cfg, max_pages=None):
    """Convert PDF to position-aware OCR text using per-supplier config."""
    print(f"OCR: {os.path.basename(pdf_path)}"
          + (f" (first {max_pages} page(s))" if max_pages else ""))

    images = _pdf_to_images(pdf_path, ocr_cfg, max_pages)
    if not images:
        print("  ERROR: could not convert PDF to images.")
        return None

    pages = []
    for i, image in enumerate(images, 1):
        try:
            preprocessed = preprocess_image_pil(image, pre_cfg)
        except Exception as exc:
            print(f"  WARN: preprocessing failed on page {i}: {exc} — using greyscale.")
            preprocessed = image.convert("L")

        page_text = extract_with_real_positions(preprocessed, ocr_cfg)
        pages.append(
            f"\n=== OCR: Full Page {i} (Real Positions) ===\n"
            f"{page_text.strip()}\n"
            f"=== End of Page {i} ==="
        )

    return "\n".join(pages).strip()


def get_numbered_ocr_text(pdf_path, pre_cfg, ocr_cfg, max_pages=None):
    """Return OCR text with line numbers — useful for debugging LLM extraction."""
    raw = pdf_to_ocr_text(pdf_path, pre_cfg, ocr_cfg, max_pages)
    if not raw:
        return "ERROR: OCR failed."
    lines = raw.split("\n")
    numbered = [
        f"{i:3d}: {line}" if line.strip() else f"{i:3d}: [empty]"
        for i, line in enumerate(lines, 1)
    ]
    numbered.append(f"\nTotal lines: {len(lines)}")
    return "\n".join(numbered)


def batch_ocr_to_txt(input_dir, output_dir, pre_cfg, ocr_cfg, max_pages=None):
    """Process all PDFs in input_dir, write one .txt OCR file per PDF to output_dir."""
    os.makedirs(output_dir, exist_ok=True)
    pdf_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".pdf")]
    if not pdf_files:
        print(f"No PDF files found in: {input_dir}")
        return
    for filename in pdf_files:
        txt_path = os.path.join(output_dir, os.path.splitext(filename)[0] + ".txt")
        text = get_numbered_ocr_text(os.path.join(input_dir, filename), pre_cfg, ocr_cfg, max_pages)
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(text)
        print(f"  Saved: {txt_path}")


print("OCR functions ready")


OCR functions ready


In [42]:
batch_ocr_to_txt(INPUT_DIR, OCR_OUT_DIR, CFG["preprocess"], CFG["ocr"], max_pages=1)

OCR: 2602100643.pdf (first 1 page(s))


  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\easyflyer\2602100643.txt
OCR: 2602100770.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\easyflyer\2602100770.txt
OCR: Easy Flyer Air Freight Invoice Data Points To Be Collected.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\easyflyer\Easy Flyer Air Freight Invoice Data Points To Be Collected.txt


## LLM — Cortex Completion (prompt z pliku)

In [54]:
def ask_cortex_invoice_data(session, extracted_text, prompt_file):
    """
    Send extracted OCR text to Snowflake Cortex LLM.
    Prompt template is loaded from prompt_file — uses {extracted_text} placeholder.
    """
    try:
        with open(prompt_file, encoding="utf-8") as f:
            prompt_template = f.read()

        prompt = prompt_template.format(extracted_text=extracted_text)
        escaped = prompt.replace("'", "\\'")

        sql = f"""
            SELECT snowflake.cortex.complete(
                'llama3.1-70b',
                $$ {escaped} $$
            )
        """
        result = session.sql(sql).collect()
        return result[0][0].strip(), None
    except Exception as e:
        return None, f"❌ Cortex error: {e}"


print("ask_cortex_invoice_data() ready")

ask_cortex_invoice_data() ready


## Normalizacja pól i zapis do Snowflake

In [56]:
def _esc(val):
    if val is None or str(val).strip() in ("", "N/A", "n/a"):
        return "NULL"
    return "'" + str(val).replace("'", "''") + "'"


def _parse_num(s):
    try:
        return float(str(s).replace(",", "").replace(".", "."))
    except Exception:
        return None


# def _convert_date(date_str):
#     """Convert MM/DD/YYYY to DD/MM/YYYY. Returns original if not matching."""
#     m = re.match(r"^(\d{1,2})/(\d{1,2})/(\d{4})$", str(date_str).strip())
#     if not m:
#         return date_str
#     mm, dd, yyyy = m.groups()
#     return f"{dd.zfill(2)}/{mm.zfill(2)}/{yyyy}"


# ── All known reference type columns (must match DDL) ─────────────────────────
_REF_COLS = [
    "MAWB", "TMS", "REQNA", "SHIPMENT_NUMBER", "PTN",
    "TRACKING_ID", "AUTOLIV_PERSON_CONCERNED", "ORDER_REFERENCES",
    "REFERENCE_NUMBER", "TMS_NUMBER", "AWB", "HAWB",
]


def normalize_invoice(raw_json, field_map):
    """
    Map supplier-specific JSON keys to a canonical internal schema using field_map from config.
    NOTE: Date conversion is handled in the prompts (LLM outputs DD/MM/YYYY directly).
          Do NOT re-convert here — that would swap day/month a second time.
    """
    inv = {}
    for canonical, src_key in field_map.items():
        inv[canonical] = raw_json.get(src_key, "N/A")

    # normalise dates
    # for date_field in ("invoice_date", "due_date", "ship_date", "delivery_date"):
    #     if date_field in inv:
    #         inv[date_field] = _convert_date(inv[date_field])

    # pass through all original keys as well (needed for references lookup)
    for k, v in raw_json.items():
        if k not in inv:
            inv[k] = v

    inv["_raw"] = raw_json
    return inv


def build_references(inv, ref_cfg):
    """
    Build list of (ref_type, ref_value) from config references[] definition.
    Special type PTN_OR_ORDER_REF: detects PTN pattern or falls back to ORDER_REFERENCES.
    """
    refs = []
    for ref_def in ref_cfg:
        ref_type = ref_def["type"]
        val = inv.get(ref_def["json_key"])

        if ref_type == "PTN_OR_ORDER_REF":
            raw = str(val or "").strip()
            if not raw or raw in ("N/A", "n/a"):
                refs.append(("ORDER_REFERENCES", None))
                continue
            ptn_match = re.search(r"(PTN[-\s]?\w+)", raw, re.IGNORECASE)
            if ptn_match:
                refs.append(("PTN", ptn_match.group(1).strip()))
                remainder = raw.replace(ptn_match.group(0), "").strip().strip(",").strip()
                if remainder:
                    refs.append(("ORDER_REFERENCES", remainder))
            else:
                refs.append(("ORDER_REFERENCES", raw))
        else:
            refs.append((ref_type, val))

    return refs


def insert_invoices_to_staging(
    session,
    normalized_invoices,
    ref_cfg,
    header_table="STG_INVOICE_HEADER",
    charge_table="STG_INVOICE_CHARGE_LINES",
    ref_table="STG_INVOICE_REFERENCES",
):
    if not normalized_invoices:
        print("No data to insert.")
        return

    existing = {
        r[0]
        for r in session.sql(f"SELECT INVOICE_ID FROM {header_table}").collect()
        if r[0]
    }
    print(f"Already in {header_table}: {len(existing)} invoice(s)")

    inserted_h = inserted_c = inserted_r = 0

    for inv in normalized_invoices:
        invoice_id = str(inv.get("invoice_id") or "").strip()
        if not invoice_id or invoice_id == "N/A":
            print(f"  SKIP (no invoice_id): {inv.get('filename', '?')}")
            continue
        if invoice_id in existing:
            print(f"  SKIP (duplicate): {invoice_id}")
            continue

        weight_val = _parse_num(inv.get("weight_value"))
        amount_val = _parse_num(inv.get("amount"))

        line_items = inv.get("line_items") or []
        if isinstance(line_items, str):
            try:
                line_items = json.loads(line_items)
            except Exception:
                line_items = []

        ocr_escaped = "'" + json.dumps(inv.get("_raw", inv), ensure_ascii=False).replace("'", "''") + "'"

        # ── STG_INVOICE_HEADER ────────────────────────────────────────────
        session.sql(f"""
            INSERT INTO {header_table} (
                INVOICE_ID, INVOICE_DATE, DUE_DATE,
                SHIP_DATE, DELIVERY_DATE, DELIVERY_TIME, DELIVERY_TIMEZONE,
                CARRIER_NAME_RAW, BILL_TO_RAW,
                INVOICE_TOTAL, CURRENCY,
                TRANSPORT_MODE,
                SHIP_FROM, SHIP_TO,
                WEIGHT_VALUE, WEIGHT_UNIT,
                SOURCE_CARRIER, OCR_RAW_PAYLOAD,
                LOAD_TIMESTAMP, PROCESSING_STATUS
            )
            SELECT
                {_esc(invoice_id)},
                TRY_TO_DATE({_esc(inv.get("invoice_date"))}, 'DD/MM/YYYY'),
                NULL,
                TRY_TO_DATE({_esc(inv.get("ship_date"))}, 'DD/MM/YYYY'),
                NULL,
                {_esc(inv.get("delivery_time"))},
                {_esc(inv.get("delivery_timezone"))},
                {_esc(inv.get("supplier"))},
                {_esc(inv.get("bill_to"))},
                {amount_val if amount_val is not None else "NULL"},
                {_esc(inv.get("currency"))},
                'Air',
                {_esc(inv.get("ship_from"))},
                {_esc(inv.get("ship_to"))},
                {weight_val if weight_val is not None else "NULL"},
                {_esc(inv.get("weight_unit"))},
                {_esc(inv.get("supplier"))},
                PARSE_JSON({ocr_escaped}),
                CURRENT_TIMESTAMP(),
                'NEW'
        """).collect()
        inserted_h += 1

        # ── STG_INVOICE_CHARGE_LINES ──────────────────────────────────────
        for seq, item in enumerate(line_items, 1):
            if not isinstance(item, dict):
                continue
            session.sql(f"""
                INSERT INTO {charge_table} (
                    CHARGE_LINE_ID, INVOICE_ID, SOURCE_CARRIER,
                    CHARGE_SEQUENCE, CHARGE_DESCRIPTION_RAW,
                    CHARGE_QUANTITY, CHARGE_UNIT_PRICE,
                    CHARGE_AMOUNT, CURRENCY
                ) VALUES (
                    {_esc(str(uuid.uuid4()))},
                    {_esc(invoice_id)},
                    {_esc(inv.get("supplier"))},
                    {seq},
                    {_esc(item.get("description"))},
                    {_parse_num(item.get("quantity")) if _parse_num(item.get("quantity")) is not None else "NULL"},
                    {_parse_num(item.get("unit_price")) if _parse_num(item.get("unit_price")) is not None else "NULL"},
                    {_parse_num(item.get("amount")) if _parse_num(item.get("amount")) is not None else "NULL"},
                    {_esc(item.get("currency") or inv.get("currency"))}
                )
            """).collect()
            inserted_c += 1

        # ── STG_INVOICE_REFERENCES (pivoted — one row per invoice) ────────
        # Step 1: fill from ref_cfg rules
        ref_dict = {}
        for ref_type, ref_val in build_references(inv, ref_cfg):
            if ref_val is not None:
                ref_dict[ref_type.upper()] = ref_val

        # # Step 2: fallback — pick up any _REF_COLS field the LLM extracted
        # # directly (e.g. "mawb", "hawb") that ref_cfg doesn't cover
        # for col in _REF_COLS:
        #     if col not in ref_dict:
        #         val = inv.get(col.lower())
        #         if val and str(val).strip() not in ("", "N/A", "n/a"):
        #             ref_dict[col] = str(val).strip()

        session.sql(f"""
            INSERT INTO {ref_table} (
                REFERENCE_ID, INVOICE_ID, SOURCE_CARRIER,
                {", ".join(_REF_COLS)}
            ) VALUES (
                {_esc(str(uuid.uuid4()))},
                {_esc(invoice_id)},
                {_esc(inv.get("supplier"))},
                {", ".join(_esc(ref_dict.get(col)) for col in _REF_COLS)}
            )
        """).collect()
        inserted_r += 1

        existing.add(invoice_id)
        print(f"  OK  {invoice_id}")

    print(f"\nSTG_INVOICE_HEADER:       {inserted_h} row(s)")
    print(f"STG_INVOICE_CHARGE_LINES: {inserted_c} row(s)")
    print(f"STG_INVOICE_REFERENCES:   {inserted_r} row(s)")


print("Staging functions ready")

Staging functions ready


## [DEBUG] Batch OCR → pliki .txt (opcjonalnie, do podglądu)

In [57]:
# Uruchom by zapisać surowy OCR do plików .txt — przydatne przy tuningowaniu promptów
batch_ocr_to_txt(
    input_dir  = INPUT_DIR,
    output_dir = OCR_OUT_DIR,
    pre_cfg    = CFG["preprocess"],
    ocr_cfg    = CFG["ocr"],
    max_pages  = CFG["max_pages"],
)

OCR: ARInvoice-FCUS-624688.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\xpd\ARInvoice-FCUS-624688.txt
OCR: ARInvoice-FCUS-626918.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\xpd\ARInvoice-FCUS-626918.txt
OCR: ARInvoice-FCUS-626985.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\xpd\ARInvoice-FCUS-626985.txt
OCR: ARInvoice-FCUS-627285.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\xpd\ARInvoice-FCUS-627285.txt
OCR: ARInvoice-FCUS-627506.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\xpd\ARInvoice-FCUS-627506.txt
OCR: Europartners XPD Vendor Invoice Data Points.pdf (first 1 page(s))
  Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\xpd\Europartners XPD Vendor Invoice Data Points.txt


## Główny pipeline — OCR + LLM → surowe dane

In [58]:
def process_directory(input_dir, pre_cfg, ocr_cfg, prompt_file, field_map, max_pages=1):
    """
    Full pipeline for one supplier:
      1. PDF → OCR text  (per-supplier preprocess + OCR config)
      2. OCR text → LLM JSON  (prompt from file)
      3. Normalize fields  (field_map from config)
      4. Return list of normalized invoice dicts
    """
    print(f"Scanning: {input_dir}")
    print("=" * 60)

    raw_invoices   = []
    normalized_all = []

    for filename in sorted(os.listdir(input_dir)):
        if not filename.lower().endswith(".pdf"):
            continue

        print(f"\nProcessing: {filename}")
        extracted_text = pdf_to_ocr_text(
            os.path.join(input_dir, filename), pre_cfg, ocr_cfg, max_pages=max_pages
        )
        if not extracted_text:
            print(f"  ERROR: OCR returned nothing for {filename}")
            continue

        structured_output, error = ask_cortex_invoice_data(session, extracted_text, prompt_file)
        if error:
            print(f"  ERROR (Cortex): {error}")
            continue

        # strip possible markdown code fences
        clean_output = structured_output.strip().strip("`").strip()
        if clean_output.startswith("json"):
            clean_output = clean_output[4:].strip()

        try:
            invoices = json.loads(clean_output)
        except json.JSONDecodeError as e:
            print(f"  ERROR (JSON parse): {e}\n  Raw: {structured_output[:300]}")
            continue

        for inv_raw in invoices:
            inv_raw["filename"] = filename
            normalized = normalize_invoice(inv_raw, field_map)
            normalized["filename"] = filename
            normalized_all.append(normalized)
            raw_invoices.append(inv_raw)

        print("✅ Extracted:")
        print(json.dumps(invoices, indent=2, ensure_ascii=False))

    print("=" * 60)
    print(f"Total invoices extracted: {len(normalized_all)}")
    return raw_invoices, normalized_all


# ── URUCHOM ───────────────────────────────────────────────────────────────────
raw_invoices_data, normalized_invoices_data = process_directory(
    input_dir   = INPUT_DIR,
    pre_cfg     = CFG["preprocess"],
    ocr_cfg     = CFG["ocr"],
    prompt_file = PROMPT_FILE,
    field_map   = CFG["field_map"],
    max_pages   = CFG["max_pages"],
)

Scanning: C:\Users\jakub.klosowski\OneDrive - Autoliv\Desktop\Invoices\Xpd

Processing: ARInvoice-FCUS-624688.pdf
OCR: ARInvoice-FCUS-624688.pdf (first 1 page(s))
✅ Extracted:
[
  {
    "vendor": "Europartners Mexico",
    "invoice_number": "FCUS-624688",
    "bill_to": "Autoliv ASP, Inc.     3350 Airport Road     Ogden, UT 84405   USA",
    "invoice_date": "20/01/2026",
    "ship_date": "15/01/2026",
    "shipper": "EICHSFELDER SCHRAUBENWERK GMBH, Rengelréder Weg 13 Heilbad Heiligenstadt Thüringen 37308 Germany DE811210116",
    "consignee": "AUTOLIV AST DEPOT / CASAS INT 9355 Airway Road. suite 4 San Diego California 92154 United States",
    "tms_number": "N/A",
    "ptn_number": "S2091-F5F4-1GQYL",
    "autoliv_person_concerned": "Sirenia Tirado",
    "weight_value": "4.314",
    "weight_unit": "KG",
    "currency": "USD",
    "line_items": [
      {
        "code": "81141601",
        "description": "Air Freight",
        "quantity": "0",
        "unit_price": "0.00",
        "tax

## Zapis do Snowflake Staging

In [59]:
session.sql("USE DATABASE DEV_SCM_DB").collect()
session.sql("USE SCHEMA OUT_TMS").collect()

insert_invoices_to_staging(
    session            = session,
    normalized_invoices = normalized_invoices_data,
    ref_cfg            = CFG["references"],
)

Already in STG_INVOICE_HEADER: 13 invoice(s)
  OK  FCUS-624688
  OK  FCUS-626918
  OK  FCUS-626985
  OK  FCUS-627285
  OK  FCUS-627506
  OK  FCUS-620207

STG_INVOICE_HEADER:       6 row(s)
STG_INVOICE_CHARGE_LINES: 15 row(s)
STG_INVOICE_REFERENCES:   6 row(s)


In [ ]:
def export_to_excel(raw_invoices, output_path=None):
    """
    Export raw invoice data to Excel — two representations in one file:
      Sheet 1 "Invoices"   : one row per invoice, scalar fields + line_items_json column (JSON string)
      Sheet 2 "Line Items" : one row per charge line (expanded), with invoice_no as key
    Filename includes supplier name suffix.
    """
    if not raw_invoices:
        print("No data to export.")
        return

    supplier_name = CFG.get("supplier_name", SUPPLIER).replace(" ", "_").replace(",", "").replace(".", "")
    if output_path is None:
        output_path = os.path.join(OCR_OUT_DIR, f"invoices_{supplier_name}.xlsx")

    # ── Sheet 1: header rows + line_items_json column ─────────────────────────
    header_rows = []
    for inv in raw_invoices:
        row = {k: v for k, v in inv.items() if k != "line_items"}
        line_items = inv.get("line_items") or []
        row["line_items_json"] = json.dumps(line_items, ensure_ascii=False) if line_items else "[]"
        header_rows.append(row)

    df_headers = pd.DataFrame(header_rows)

    # ── Sheet 2: line items expanded ──────────────────────────────────────────
    line_rows = []
    for inv in raw_invoices:
        invoice_id = inv.get("invoice_no") or inv.get("invoice_number") or inv.get("filename", "?")
        for item in (inv.get("line_items") or []):
            if isinstance(item, dict):
                line_rows.append({"invoice_no": invoice_id, **item})

    df_lines = pd.DataFrame(line_rows) if line_rows else pd.DataFrame()

    # ── Write to Excel ────────────────────────────────────────────────────────
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df_headers.to_excel(writer, sheet_name="Invoices", index=False)
        if not df_lines.empty:
            df_lines.to_excel(writer, sheet_name="Line Items", index=False)

    print(f"Saved: {output_path}")
    print(f"  Invoices sheet   : {len(df_headers)} row(s), {len(df_headers.columns)} columns (incl. line_items_json)")
    if not df_lines.empty:
        print(f"  Line Items sheet : {len(df_lines)} row(s) (expanded)")
    return output_path


export_to_excel(raw_invoices_data)


Saved: c:\Users\jakub.klosowski\email\FAP\unified\ocr_output\courier_network\invoices_Courier_Network_Inc.xlsx
  Invoices sheet   : 4 row(s), 22 columns (incl. line_items_json)
  Line Items sheet : 6 row(s) (expanded)


'c:\\Users\\jakub.klosowski\\email\\FAP\\unified\\ocr_output\\courier_network\\invoices_Courier_Network_Inc.xlsx'